In [1]:
from pathlib import Path
from typing import Mapping

import dask.array as da
import xarray as xr
from xarray import DataArray, Dataset, DataTree
from spatialdata import SpatialData
from spatialdata.transformations import Scale, Identity


def _open_wsi(path: str | Path) -> tuple[str, xr.Dataset, object, dict]:
    import tiffslide

    path = Path(path)
    image_name = path.stem
    slide = tiffslide.open_slide(str(path))
    zarr_store = slide.zarr_group.store
    zarr_img = xr.open_zarr(zarr_store, consolidated=False, mask_and_scale=False)

    metadata = {
        "properties": slide.properties,
        "dimensions": slide.dimensions,
        "level_count": slide.level_count,
        "level_dimensions": slide.level_dimensions,
        "level_downsamples": slide.level_downsamples,
    }
    return image_name, zarr_img, slide, metadata


def load_cell_dive_spatialdata_tiffslide(
    image_files: Mapping[str, str],  # {channel_name: image_path}
    chunks: tuple[int, int, int] = (1, 1024, 1024),
    coordinate_system: str = "global",
) -> SpatialData:
    """Load Cell DIVE multi-channel multiscale image as SpatialData using tiffslide backend."""

    channel_names = list(image_files.keys())
    paths = list(image_files.values())

    pyramids = []
    base_shape = None
    slide_downsamples = None

    for path in paths:
        _, img, slide, _ = _open_wsi(path)

        levels = [img[str(k)] for k in sorted(img.keys(), key=int)]
        if base_shape is None:
            base_shape = levels[0].shape[-2:]
            slide_downsamples = slide.level_downsamples

        pyramids.append(levels)

    num_levels = len(pyramids[0])
    assert all(len(p) == num_levels for p in pyramids), "All images must have the same number of pyramid levels"

    multiscale_arrays = [da.stack([p[i] for p in pyramids], axis=0) for i in range(num_levels)]

    images = {}
    for level, arr in enumerate(multiscale_arrays):
        data = DataArray(arr, dims=("c", "y", "x")).chunk({"c": chunks[0], "y": chunks[1], "x": chunks[2]})
        scale_y = base_shape[0] / arr.shape[-2]
        scale_x = base_shape[1] / arr.shape[-1]
        transform = (
            Scale([scale_y, scale_x], axes=("y", "x"))
            if (scale_y != 1.0 or scale_x != 1.0)
            else Identity()
        )
        data.coords["c"] = channel_names
        data.attrs["transform"] = {coordinate_system: transform}
        images[f"scale{level}"] = Dataset({"image": data})

    tree = DataTree.from_dict(images)
    return tree



C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa_v2\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa_v2\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
from pathlib import Path

def build_image_dict_from_folder(folder: str | Path) -> dict[str, str]:
    """
    Builds a dictionary mapping 'R<round>_<channel>' to file paths.

    Expected filename format:
        SLIDE-XXXX_<round>.0.X_R000_<dye>_<channel>_...ome.tif[f]
    DAPI may have no <channel>, so handled as a special case.

    Args:
        folder: Directory containing OME-TIFF files.

    Returns:
        Dictionary like {'R1_CD4': '/path/to/file.tiff', 'R1_DAPI': '/path/to/file.tif'}
    """
    folder = Path(folder)
    image_dict = {}

    for file in folder.iterdir():
        if not file.is_file():
            continue
        if not file.suffix.lower().endswith(("tif", "tiff")):
            continue

        parts = file.stem.split("_")
        try:
            # Round number from the 2nd part (e.g., 1 from "1.0.4")
            round_str = parts[1]
            round_num = round_str.split(".")[0]

            dye = parts[3]
            channel = parts[4] if len(parts) > 4 and parts[4] else None

            # Special case: DAPI has no channel part
            if dye.upper() == "DAPI" or not channel:
                channel_name = "DAPI"
            else:
                channel_name = channel

            key = f"R{round_num}_{channel_name}"
            image_dict[key] = str(file.resolve())

        except (IndexError, ValueError):
            print(f"⚠️ Skipping file with unrecognized structure: {file.name}")
            continue

    return image_dict


In [11]:
folder_path = r"C:\Analysis\Data\SLIDE-0329"
image_dict = build_image_dict_from_folder(folder_path)
(image_dict)

NameError: name 'sopa' is not defined

In [4]:
# Load channels as lazy dask arrays
image = load_cell_dive_spatialdata_tiffslide(image_dict, chunks = (1, 512, 512), coordinate_system = "SLIDE-0329")
sdata = SpatialData(images={"SLIDE-0329": image})

In [5]:
sdata

SpatialData object
└── Images
      └── 'SLIDE-0329': DataTree[cyx] (15, 62617, 66406), (15, 31308, 33203), (15, 15654, 16601), (15, 7827, 8300), (15, 3913, 4150), (15, 1956, 2075), (15, 978, 1037), (15, 489, 518), (15, 244, 259), (15, 122, 129)
with coordinate systems:
    ▸ 'SLIDE-0329', with elements:
        SLIDE-0329 (Images)

In [6]:
sdata["SLIDE-0329"]["scale0"]["image"]

<xarray.DataArray 'image' (c: 15, y: 62617, x: 66406)> Size: 125GB
dask.array<rechunk-merge, shape=(15, 62617, 66406), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U20 1kB 'R1_Bit2-RS0584-Cy3B' ... 'R3_Bit9-RS0805-488'
Dimensions without coordinates: y, x
Attributes:
    transform:  {'SLIDE-0329': Identity }

In [7]:
#sdata.rename_coordinate_systems({'global':'SLIDE-0272'})

In [8]:
sdata.write("SLIDE_0329.zarr", overwrite=True)

INFO     The Zarr backing store has been changed from None the new file path: SLIDE_0329.zarr                      


In [9]:
import napari
from napari_spatialdata import Interactive
viewer = napari.Viewer()
interactive = Interactive(sdata, viewer)

2026-03-30 23:29:17.677 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.
2026-03-30 23:29:25.854 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:29:25.854 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:29:41.442 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:29:41.442 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:31:17.388 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:31:17.391 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:31:17.392 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:31:24.663

INFO: Layer(s) inherited info from SLIDE-0329
INFO: Layer saved


2026-03-30 23:34:24.794 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:34:24.794 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:34:25.261 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually save to disk.


INFO: Layer(s) inherited info from SLIDE-0329
INFO: Layer saved


2026-03-30 23:34:34.042 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:34:34.055 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:35:09.793 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:35:09.796 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:35:45.433 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:35:45.433 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:36:43.247 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually save to disk.


INFO: Layer(s) inherited info from SLIDE-0329
INFO: Layer saved


2026-03-30 23:36:44.514 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:36:44.514 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:36:45.294 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually save to disk.


INFO: Layer(s) inherited info from SLIDE-0329
INFO: Layer saved


2026-03-30 23:36:47.107 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:36:47.107 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-03-30 23:36:47.455 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually save to disk.


INFO: Layer(s) inherited info from SLIDE-0329
INFO: Layer saved


In [13]:
sdata

SpatialData object, with associated Zarr store: C:\Analysis\M11_guidepool\SLIDE_0329.zarr
├── Images
│     └── 'SLIDE-0329': DataTree[cyx] (15, 62617, 66406), (15, 31308, 33203), (15, 15654, 16601), (15, 7827, 8300), (15, 3913, 4150), (15, 1956, 2075), (15, 978, 1037), (15, 489, 518), (15, 244, 259), (15, 122, 129)
└── Shapes
      ├── 'tumor_B5_A_1L': GeoDataFrame shape: (1, 1) (2D shapes)
      ├── 'tumor_B5_A_1R': GeoDataFrame shape: (1, 1) (2D shapes)
      ├── 'tumor_B5_A_1R1L': GeoDataFrame shape: (1, 1) (2D shapes)
      ├── 'tumor_B5_A_2R': GeoDataFrame shape: (1, 1) (2D shapes)
      └── 'tumor_B5_A_NH': GeoDataFrame shape: (2, 1) (2D shapes)
with coordinate systems:
    ▸ 'SLIDE-0329', with elements:
        SLIDE-0329 (Images), tumor_B5_A_1L (Shapes), tumor_B5_A_1R (Shapes), tumor_B5_A_1R1L (Shapes), tumor_B5_A_2R (Shapes), tumor_B5_A_NH (Shapes)

In [12]:
#sdata.write_element(['tumor_B5_A_1L','tumor_B5_A_1R','tumor_B5_A_1R1L','tumor_B5_A_2R','tumor_B5_A_NH'])

In [18]:
from spatialdata import SpatialData
import napari

# Load SpatialData object
sdata = SpatialData.read("test_spatialdata_v3.zarr")

# Access the image DataTree and extract multiscale Dask arrays
tree = sdata.images["CellDIVE"]
multiscale = [tree[f"scale{level}"].image.data for level in range(len(tree.children))]


# Extract channel names from coordinates
channel_names = tree["scale0"].image.coords["c"].values.tolist()


viewer = napari.Viewer()
# Launch napari with multiscale image (channel names only as metadata or printout)
for i, name in enumerate(channel_names):
    channel_data = [level[i] for level in multiscale]  # collect ith channel across scales
    viewer.add_image(
        channel_data,
        multiscale=True,
        name=name,
        scale=(1, 1),  # or pixel size if needed
        blending="additive"
    )

version mismatch: detected: RasterFormatV02, requested: FormatV04
C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
C:\Users\ratnayn\AppData\Local\miniconda3\envs\sopa\Lib\site-packa